# Grid Measure — **Id-Vg 전달특성(Transfer Curve)** 자동 측정

[grid_measure_idvd.ipynb](grid_measure_idvd.ipynb) (출력특성 Id-Vd) 를 **전달특성 Id-Vg** 버전으로 바꾼 파일.
S300 좌표이동·B1500 제어 코드는 **원본과 완전히 동일**하고, 바뀐 것은 **어느 채널을 sweep 하느냐** 뿐이다.

| | Id-Vd (원본, output curve) | **Id-Vg (이 파일, transfer curve)** |
|---|---|---|
| VAR1 (주 sweep) | Vd — SMU2 / Drain | **Vg — SMU1 / Gate** |
| VAR2 (부 sweep) | Vg — SMU1 / Gate | **Vd — SMU2 / Drain** |
| Source (SMU3) | 0 V 고정 | 0 V 고정 (동일) |
| 저장 폴더 | `results_idvd/` | **`results_idvg/`** |

즉 `SMU_CONFIG` 에서 **drive="sweep" 을 SMU1(Gate)로 옮기고**, `VAR1_CONFIG`/`VAR2_CONFIG` 의
unit·전압범위를 맞바꾼 것이 전부다. 측정 함수(`measure_iv_full`, `measure_iv_fast`)는 손대지 않았다.

> 화면 ↔ 코드 매핑
> | EasyEXPERT UI | 코드 |
> |---|---|
> | VAR1 (Unit/Name/Direction/Linear-Log/Start/Stop/Step/No of Step/Compliance/Pwr Comp) | `VAR1_CONFIG` |
> | VAR2 (부 sweep, 각 스텝마다 VAR1 반복) | `VAR2_CONFIG` (외부 루프로 구현) |
> | Timing (Hold/Delay/* Sweep) | `TIMING_CONFIG` |
> | Constants (Source/Compliance) | `SMU_CONFIG` 의 `const_v` |
> | Set Ranging Mode (Mode/Range) | `RANGE_CONFIG` |


## ⚠️ 코드 실행 전 — 사람이 손으로 끝내놔야 하는 준비 (Nucleus UI)
매뉴얼 워크플로우 중 이 단계들은 **코드가 대체하지 않으므로** 먼저 수동으로 완료해야 함:
1. 척 로드 / 진공 ON / 소자 세팅
2. **Alignment** (2-point align) — 안 하면 좌표가 통째로 어긋남
3. **Tipping / Set Contact** — 팁 contact 높이를 잡아둠
4. **첫 소자에 팁을 직접 contact** 시킨 상태로 둠 → 아래 5번 셀에서 그 위치를 원점·contact높이로 등록

> 측정 중 light off 등은 Nucleus 쪽 물리 작업이라 코드 밖.

In [ ]:
import os
import time
import pandas as pd
import pyvisa
from pymeasure.instruments.agilent import AgilentB1500

# --- pandas 3.0 호환 shim -----------------------------------------------------
# pandas 2.1 에서 DataFrame.applymap -> DataFrame.map 으로 이름 바뀌고
# pandas 3.0 에서 applymap 이 제거됨. 그런데 pymeasure 0.16 의 B1500 read_data 가
# 아직 applymap 을 써서 측정 데이터 읽을 때 에러남. map 이 applymap 과 동작 동일하므로
# 옛 이름을 다시 연결해 호환을 살린다. (라이브러리가 pandas 3 지원하면 삭제 가능)
if not hasattr(pd.DataFrame, "applymap"):
    pd.DataFrame.applymap = pd.DataFrame.map

## 0. 설정 (여기만 바꾸면 됨)

In [ ]:
# --- VISA 구현체 지정 ---------------------------------------------------
# 이 PC 에는 NI-VISA 와 Keysight VISA 가 둘 다 설치되어 있다. 인자 없이 부르는
# pyvisa.ResourceManager() 는 공용 visa32.dll(IVI 라우터)를 타고 Keysight VISA 로
# 붙는다. 그런데 Keysight 82357B 어댑터는 빠져 있어서(CM_PROB_PHANTOM)
# GPIB 장비를 못 찾고 VI_ERROR_RSRC_NFOUND 가 난다. 실제로 꽂혀 있는 건
# NI GPIB-USB-HS 이므로 NI-VISA 를 명시적으로 지정한다.
VISA_LIB = r"C:\Windows\System32\nivisa64.dll"

# --- 장비 주소 ---
S300_GPIB  = "GPIB0::28::INSTR"
B1500_GPIB = "GPIB0::17::INSTR"

# --- 좌표 CSV (형식: Subsite Name, X Position, Y Position, Note), 단위 = micron ---
# 테스트용(3점, 최대 100um 이동). 실제 측정 시 아래 old 경로로 되돌릴 것.
# COORD_CSV = "../../utils/test_coordinates.csv"   # 테스트용(3점)
COORD_CSV = r"C:\Users\UNL_microscope\Desktop\semi-auto\utils\grid_4x4.csv"     # 실제 측정용 (4x4 = 16점)

# COORD_CSV = "../../utils/old/grid_coordinates.csv"   # 실제 측정용 (160점)

# --- [구버전] 원본 measure_iv (## 6 루프) 전용 파라미터 ----------------------
# Id-Vg 측정은 ## 0-B 의 VAR1_CONFIG/VAR2_CONFIG 를 쓰고 ## 6-B 루프로 돌린다.
# 아래 4개는 ## 6 원본 루프(measure_iv)만 참조하므로 지금 셋업에선 사용되지 않음.
# (원본 measure_iv 는 drive="sweep" 채널을 sweep 하므로, 이 파일에선 자동으로 Gate 를 쓴다)
V_START      = -10.0   # 시작 전압 [V]  (Vg)
V_STOP       = 20.0    # 끝 전압 [V]    (Vg)
V_POINTS     = 31      # sweep 포인트 수 (1 V 간격)
I_COMPLIANCE = 1e-3    # 전류 컴플라이언스 [A]

# --- 팁(프로브) ↔ SMU ↔ 역할 매핑 -------------------------------------------
# 현재 셋업 = Id-Vg 전달특성 (transfer curve)
#   VAR1 = Vg sweep (SMU1 / Gate)   ← 원본 Id-Vd 와 반대. 주 sweep 이 게이트다.
#   VAR2 = Vd step  (SMU2 / Drain)  ← drive="const" 로 두고 VAR2_CONFIG 가 스텝마다 덮어씀
#                                     보통 linear 영역(0.1 V) + 포화 영역(10 V) 2개만 찍는다.
#   Source(SMU3) = 0 V 고정
#
# [주의] smuN 은 '장착 슬롯 번호'가 아니다. 실측 확인 결과(b1500.smuN.channel):
#   smu1 -> 본체 슬롯 2 (B1517A HRSMU)   ← Gate 프로브
#   smu2 -> 본체 슬롯 3 (B1517A HRSMU)   ← Drain 프로브
#   smu3 -> 본체 슬롯 4 (B1517A HRSMU)   ← Source 프로브
#   smu4 -> 없음  (슬롯 1 이 B1530A WGFMU 라 SMU 번호가 한 칸 밀림)
#
#   role       : 표기용 이름 (자유)
#   drive      : "sweep" → VAR1 주 sweep 채널 (지금은 Drain)
#                "const" → DC 고정. VAR2 채널도 반드시 여기 속해야 한다.
#                "off"   → 사용 안 함(disable)
#   const_v    : drive="const" 일 때 인가 전압 [V]
#                (VAR2 채널은 VAR2_CONFIG 의 스텝 값으로 덮어써지므로 초기값 의미만 가짐)
#   compliance : 채널별 전류 컴플라이언스 [A]  ← measure_iv_full 만 사용
SMU_CONFIG = {
    1: {"role": "GATE",   "drive": "sweep", "const_v": None, "compliance": 1e-3},     # VAR1 (주 sweep)
    2: {"role": "DRAIN",  "drive": "const", "const_v": 10.0, "compliance": 10e-3},    # VAR2 (부 sweep)
    3: {"role": "SOURCE", "drive": "const", "const_v": 0.0,  "compliance": 100e-3},
    4: {"role": "BULK",   "drive": "off",   "const_v": 0.0,  "compliance": None},
}
# ※ GATE compliance 는 1 mA 로 낮춰뒀다. 전달특성은 게이트를 크게 흔들기 때문에
#   게이트 절연막이 깨지면(Ig 폭주) 바로 소자가 죽는다. 10 mA 로 두면 보호가 안 된다.

# --- S300 척(chuck) Device ID ---
# 매뉴얼 p.37: 2 = Elite/Summit/S300/Alessi 의 척 제어 device ID (축 번호 아님)
CHUCK_ID = 2

# --- 결과 저장 폴더 ---
OUT_DIR = "results_idvg_legacy"

## 0-B. B1500 Measurement Setup 파라미터 (스크린샷 UI ↔ 코드)

위 `0.` 설정은 **원본 그대로**(원본 `measure_iv`/`measure_sampling` 용). 
아래는 **추가된 설정판** — EasyEXPERT 의 *Measurement Setup* 탭 손잡이를 그대로 노출한다.
이 값들은 아래 `## 2-B` 의 `measure_iv_full()` 이 사용한다. (원본 함수엔 영향 없음)

> **Id-Vd 원본과 다른 곳은 딱 여기 + `SMU_CONFIG` 두 군데뿐이다.** VAR1=Vg, VAR2=Vd 로 뒤집혀 있다.

In [ ]:
# ============================================================================
# B1500 "Measurement Setup" 화면을 코드로 재현하는 설정값들
#   VAR1  : 주 sweep 소스   (각 VAR2 스텝마다 이 sweep 전체가 1회 돈다)
#   VAR2  : 부 sweep 소스   (지정하면 2차원 sweep)
#   Timing: Hold/Delay/Sweep(auto abort)
#   ADC   : 적분/분해능
#   Range : 채널별 측정 레인지 모드 (Set Ranging Mode)
#   Const : DC 고정 소스는 위 SMU_CONFIG 의 const_v / compliance 를 사용
#
# ▼ 현재 값 = Id-Vg 전달특성 (transfer curve)
#   VAR1 : Vg -10 -> 20 V, step 0.25 V (121점), Double sweep -> 곡선당 242점
#          Double = 정방향/역방향 왕복 -> 히스테리시스(Vth shift)를 그대로 볼 수 있음
#   VAR2 : Vd 0.1 V (linear 영역) / 10 V (포화 영역) 2 스텝
#   총 242 x 2 = 484 점
# ============================================================================

# --- VAR1 (주 sweep) : Gate 전압 -------------------------------------------
VAR1_CONFIG = {
    "unit":       1,          # sweep 할 SMU 채널 = SMU_CONFIG 의 키 (drive="sweep" 여야 함) → SMU1/Gate
    "name":       "Vg",       # 표기용 (UI Name)
    "direction":  "double",   # "single"=편도 | "double"=왕복(-10→20→-10). 히스테리시스 보려면 double
    "spacing":    "linear",   # "linear" | "log"
    "start":      -10.0,      # Start [V]  — off 영역까지 충분히 내려가야 Ioff/SS 가 잡힌다
    "stop":       20.0,       # Stop  [V]  — on 영역까지
    "points":     121,        # No of Step  (-10~20 V 를 0.25 V 간격 → 121점)
    "step":       None,       # Step [V]. 지정하면 points 대신 이걸로 계산 (0.25 넣어도 동일)
    "compliance": 1e-3,       # Compliance [A] = 1 mA  ← 게이트 보호용 (Ig 폭주 차단)
    "power_comp": None,       # Pwr Comp [W] (None=OFF)
}

# --- VAR2 (부 sweep) : Drain 전압. 필요 없으면 None ------------------------
# VAR2 채널은 SMU_CONFIG 에서 drive="const" 여야 하고, 각 스텝 값이 const_v 를 덮어쓴다.
# 전달특성은 보통 Vd 를 많이 안 찍는다. linear(작은 Vd) 1개 + saturation(큰 Vd) 1개면 충분.
#   start=0.1, stop=10.0, points=2  →  Vd = [0.1, 10.0]
# Vd 1개만 찍고 싶으면 points=1 (start 값만 사용) 또는 아예 VAR2_CONFIG = None.
VAR2_CONFIG = {
    "unit":       2,          # SMU2 / Drain
    "name":       "Vd",
    "start":      0.1,        # linear 영역
    "stop":       10.0,       # 포화 영역
    "points":     2,          # 2 스텝 → [0.1, 10.0]
    "step":       None,       # step 을 주면 등간격으로 계산됨 (지금은 points 사용)
    "compliance": 10e-3,      # Compliance [A] = 10 mA
}

# --- Timing ----------------------------------------------------------------
TIMING_CONFIG = {
    "hold":       1.0,        # Hold [s]  — VAR2 스텝마다 1 s 대기 후 VAR1 sweep 시작
    "delay":      0.0,        # Delay [s]
    "step_delay": 0.0,        # 스텝별 측정 지연 [s]. EasyEXPERT 기본(=0) 에 맞춤
    "auto_abort": False,      # "Sweep CONTINUE AT ANY" = 자동중단 안 함
    "post":       "START",    # 측정 후 출력 복귀 위치: start 값으로 복귀 후 off
}

# --- ADC (적분/분해능) -----------------------------------------------------
ADC_CONFIG = {
    "adc_type":  "HRADC",     # High Resolution ADC
    "mode":      "AUTO",      # Auto mode
    "N":         10,          # 적분 계수 10 (HRADC AUTO 기본 6 → 저노이즈 지향)
    "auto_zero": False,       # Auto-zero OFF (적분시간 절반, 기존 조건과 동일)
}

# --- Ranging Mode (Set Ranging Mode) ---------------------------------------
# 채널별 "측정 전류 레인지".
#   mode : "auto"    = Auto Ranging (range 무시)
#          "limited" = 하한 지정 자동 (지정 레인지 밑으로는 안 내려감) = UI LIMITED
#          "fixed"   = 고정 레인지                                   = UI FIXED
#   range: "1 nA","10 nA","100 nA","1 uA","10 uA",...
#
# ※ 전달특성은 off 전류(pA~nA)와 on 전류(mA)를 한 sweep 안에서 같이 본다.
#   하한을 너무 높게 잡으면 Ioff 가 바닥에 깔려 on/off ratio 와 SS 를 못 뽑는다.
#   더 낮은 Ioff 까지 보려면 "100 pA" / "10 pA" 로 내리면 되지만 그만큼 느려진다.
RANGE_CONFIG = {
    1: {"mode": "limited", "range": "1 nA"},
    2: {"mode": "limited", "range": "1 nA"},
    3: {"mode": "limited", "range": "1 nA"},
    4: {"mode": "limited", "range": "1 nA"},
}


## 1. S300 프로버 제어 (① 코드)
`S300_test_v2.ipynb` 의 `CascadeS300` 에 매뉴얼 기반 편의 함수 추가.

**기준점/안전 동작 (매뉴얼 근거)**
- `set_reference()` : 사람이 첫 소자에 contact 시킨 현재 위치를 등록 — `:set:pres 2 0 0`(현재 XY=원점, p.154 "Set Zero") + `:set:cont 2 <z>`(현재 Z=contact 높이, p.139).
- `move_xy()` : `:mov:abs 2 X Y none` — **Z를 안전높이로 자동 분리한 뒤 XY 이동**(p.56)이라 이동 중 소자가 안 긁힌다.
- `contact()`/`separate()` : `:mov:cont`/`:mov:sep` — 사람이 set한 contact 높이로 복귀/분리. 그 아래로는 안 내려가 소자를 찍지 않는다.

In [ ]:
class CascadeS300:
    def __init__(self, gpib_address=S300_GPIB):
        self.rm = pyvisa.ResourceManager(VISA_LIB)
        self.instrument = self.rm.open_resource(gpib_address)
        self.instrument.timeout = 30000      # 30 sec: 이동 완료(COMPLETE)까지 대기 (매뉴얼 권장)

    def _write(self, command):
        """메타 명령($:...) 등 '응답 없는' 명령 전용 — write 만."""
        self.instrument.write(command)

    def ask(self, command):
        """질의(? 명령) 또는 resp-on 상태의 명령. 응답 문자열 반환.
        $:set:resp on 상태에서 → 명령은 'COMPLETE', 질의는 값, 실패는 '@에러' 를 돌려줌."""
        try:
            return self.instrument.query(command).strip()
        except Exception as e:
            return f"Error: {e}"

    def cmd(self, command):
        """action/설정 명령 실행. $:set:resp on 덕에 완료되면 'COMPLETE' 반환(=완료까지 대기).
        응답이 '@' 로 시작하면 프로버 에러 → 예외 발생."""
        resp = self.ask(command)
        if resp.startswith("@"):
            raise RuntimeError(f"S300 명령 실패: {command!r} -> {resp!r}")
        return resp

    def setup(self):
        """매뉴얼 정식 원격 셋업 (Nucleus4 가이드 p.9). 순서 중요:
        $:set:resp on 을 먼저 켜야 이후 이동/설정 명령이 'COMPLETE' 응답을 줘서
        타임아웃 없이 완료를 확인할 수 있다. (이게 빠지면 모든 action 명령이 타임아웃)
        ※ 이 프로버 펌웨어는 :set: 명령에 device ID(CHUCK_ID)를 요구함(:set:unit 2 metric)."""
        self._write("$:set:mode summit")             # 명령 해석 모드 (meta, 응답없음)
        self._write("$:set:resp on")                 # ★ 명령마다 COMPLETE/에러 응답 켜기 (meta, 응답없음)
        self.cmd(":SYST:OPER:MODE REMOTE")           # 원격 모드 (device ID 불필요)
        self.cmd(f":set:unit {CHUCK_ID} metric")     # 단위 = micron (device ID 필요)
        return "OK (mode=summit, resp=on, REMOTE, metric)"

    # 하위호환 (개별 호출용)
    def set_remote(self):
        return self.cmd(":SYST:OPER:MODE REMOTE")

    def set_metric(self):
        return self.cmd(f":set:unit {CHUCK_ID} metric")   # device ID 필요

    # --- 위치 ---
    def read_position(self):
        """현재 척 좌표 (x, y, z) micron 반환 (:mov:abs? , p.62)."""
        resp = self.ask(f":mov:abs? {CHUCK_ID}")
        x, y, z = (float(v) for v in resp.replace(",", " ").split()[:3])
        return x, y, z

    def set_reference(self):
        """사람이 첫 소자에 contact 시킨 현재 위치를 원점(0,0)+contact높이로 등록.
        - 현재 XY → 원점 (:set:pres, p.154)   /   현재 Z → contact 높이 (:set:cont, p.139)"""
        x, y, z = self.read_position()
        self.cmd(f":set:pres {CHUCK_ID} 0 0")        # 현재 XY = 원점
        self.cmd(f":set:cont {CHUCK_ID} {int(z)}")   # 현재 Z = contact 높이
        print(f"기준점 등록: 현재위치 {(x, y, z)} → 원점(0,0), contact Z={int(z)}")

    # --- 분리 / 접촉 / 이동 (COMPLETE = 동작 완료. 별도 busy 폴링 불필요) ---
    def separate(self):
        """팁 분리 (:mov:sep, p.89). 완료(COMPLETE)까지 대기."""
        return self.cmd(f":mov:sep {CHUCK_ID}")

    def contact(self):
        """팁 접촉 — set된 contact 높이로 (:mov:cont, p.63). 완료까지 대기."""
        return self.cmd(f":mov:cont {CHUCK_ID}")

    def move_xy(self, dx, dy):
        """원점 기준 (dx,dy) micron 으로 XY 이동 (:mov:abs, z=none → Z 안전높이 자동분리).
        완료(COMPLETE)까지 대기 후 실제 위치 (x,y,z) 반환."""
        self.cmd(f":mov:abs {CHUCK_ID} {dx} {dy} none")
        return self.read_position()

## 2. B1500 측정 (② 코드)
`example_01.ipynb` 의 staircase sweep 을 함수로 묶고, **`SMU_CONFIG` 로 팁별 역할(sweep/const/off)을 선택**할 수 있게 함. `LINEAR_DOUBLE` = 왕복 sweep 이라 포인트는 `2*nop`.

In [ ]:
def drain_b1500_errors(b1500, max_reads=500):
    """B1500 FLEX 에러큐를 읽어서 끝까지 비운다(+0 'No Error' 나올 때까지).
    *CLS 로는 이 큐가 안 비워질 수 있어, 이전 통신오류(NCIC 등)로 쌓인
    +100 backlog 를 직접 제거한다. 비운 에러 개수를 반환."""
    n = 0
    for _ in range(max_reads):
        resp = b1500.ask("ERRX?")
        if resp.split(",")[0].strip() in ("0", "+0"):
            return n
        n += 1
    return n  # max_reads 까지 못 비우면 그대로 반환(능동 생성 의심)


def wait_measurement(b1500, timeout_s=300, poll=0.2):
    """측정이 끝날 때까지 GPIB serial poll(STB) 로 기다린다. 대기 시간[s] 반환.

    pymeasure 의 b1500.check_idle() 을 쓰면 안 된다. 그건 XE 측정이 도는 도중에
    '*OPC?' 를 GPIB 로 써넣는데, B1500 은 측정 중 들어온 질의를 제때 처리하지
    못해 응답이 오지 않는다. 증상: 에러큐는 +0(No Error) 로 깨끗하고 측정
    데이터도 안 나온 채 VISA 타임아웃까지 매달림.
    serial poll 은 출력버퍼/입력큐를 건드리지 않아 측정을 방해하지 않는다.
    STB bit4(0x10) = MAV(읽을 데이터 있음) 가 서면 완료.
    """
    c = b1500.adapter.connection
    t0 = time.time()
    while time.time() - t0 < timeout_s:
        if c.read_stb() & 0x10:
            return time.time() - t0
        time.sleep(poll)
    raise TimeoutError(
        f"측정이 {timeout_s}s 안에 끝나지 않음 (마지막 STB=0x{c.read_stb():02X})"
    )


def setup_b1500():
    """B1500 연결 및 초기화"""
    # 참고: 최신 pymeasure(0.16+)의 AgilentB1500 는 read/write_termination 을
    # 내부에서 "\r\n" 으로 자동 설정하므로 여기서 넘기면 안 됨(중복 인자 에러).
    b1500 = AgilentB1500(
        B1500_GPIB,
        # NI-488.2 는 타임아웃을 이산 단계(10/30/100/300/1000초)로 '올림'한다.
        # 600000(10분)으로 적으면 실제로는 1000초(16.7분)가 적용되어, 측정이
        # 멈췄을 때 16분 넘게 매달린다. 300000 은 정확히 300초로 적용된다.
        timeout=300000,
        visa_library=VISA_LIB,
    )
    # 연결 직후 청소: 이전 세션/통신오류(NCIC 등)로 남은 입력버퍼·에러큐를 비운다.
    b1500.clear()                      # GPIB device clear (입력버퍼/상태 리셋)
    b1500.write("*CLS")                # 표준 상태 클리어
    n = drain_b1500_errors(b1500)      # FLEX 에러큐 직접 비우기 (*CLS 로 안 비워지는 +100 backlog)
    if n:
        print(f"[setup] B1500 에러큐에서 묵은 에러 {n}개 제거함")

    b1500.initialize_all_smus()
    b1500.data_format(21, mode=1)   # SMU 초기화 후 호출
    return b1500


def measure_iv(b1500, v_start, v_stop, nop, compliance, config):
    """config(SMU_CONFIG) 에 따라 각 SMU(=팁)를 sweep/const 로 설정하고 I-V 측정.
    반환: DataFrame (포인트 2*nop, LINEAR_DOUBLE 왕복 sweep)."""
    sweep_chs = [ch for ch, c in config.items() if c["drive"] == "sweep"]
    const_chs = [ch for ch, c in config.items() if c["drive"] == "const"]
    active    = sweep_chs + const_chs
    if not sweep_chs:
        raise ValueError("sweep 할 SMU 가 없습니다. SMU_CONFIG 에서 하나는 drive='sweep' 이어야 함.")

    smus = {ch: getattr(b1500, f"smu{ch}") for ch in active}

    b1500.meas_mode("STAIRCASE_SWEEP", *[smus[ch] for ch in active])
    for ch in active:
        s = smus[ch]
        s.enable()
        s.adc_type = "HRADC"
        s.meas_range_current = "1 uA"
        s.meas_op_mode = "COMPLIANCE_SIDE"

    b1500.adc_setup("HRADC", "AUTO", 6)
    b1500.sweep_timing(0, 0.5, step_delay=0.1)        # hold, delay
    b1500.sweep_auto_abort(False, post="STOP")

    # 메인 sweep SMU (Gate 등)
    main_ch = sweep_chs[0]
    smus[main_ch].staircase_sweep_source(
        "VOLTAGE", "LINEAR_DOUBLE", "Auto Ranging",
        v_start, v_stop, nop, compliance,
    )
    # 추가로 sweep 하는 SMU 가 있으면 동기 sweep
    for ch in sweep_chs[1:]:
        smus[ch].synchronous_sweep_source("VOLTAGE", "Auto Ranging", v_start, v_stop, compliance)
    # const SMU (Drain/Source/Bulk 등) 는 DC 고정
    for ch in const_chs:
        smus[ch].ramp_source("VOLTAGE", "Auto Ranging", config[ch]["const_v"], stepsize=0.1, pause=20e-3)

    # 측정 시작
    b1500.check_errors()
    b1500.clear_buffer()
    b1500.clear_timer()
    b1500.send_trigger()

    # 끝날 때까지 대기 후 한 번에 읽기
    wait_measurement(b1500)
    data = b1500.read_data(2 * nop)

    # const SMU 0V 로 복귀
    for ch in const_chs:
        smus[ch].ramp_source("VOLTAGE", "Auto Ranging", 0, stepsize=0.1, pause=20e-3)
    return data

## 2-B. `measure_iv_full()` — Measurement Setup 을 그대로 반영한 I-V 측정

원본 `measure_iv` 는 direction/ADC/range/timing 이 **하드코딩**돼 있다. 
`measure_iv_full` 은 위 `VAR1_CONFIG / VAR2_CONFIG / TIMING_CONFIG / ADC_CONFIG / RANGE_CONFIG` 를 읽어
**UI 손잡이를 코드로 그대로 설정**한다. 역할/Constants 는 원본 `SMU_CONFIG` 를 재사용.

- **VAR1** → `staircase_sweep_source` (direction+linear/log → SweepMode, start/stop/points/compliance/Pwr Comp)
- **VAR2** → 파이썬 외부 루프 (각 스텝마다 VAR2 채널 DC 를 바꿔 VAR1 sweep 반복 → 세로로 이어붙임)
- **Timing** → `sweep_timing` + `sweep_auto_abort`
- **Ranging** → 채널별 `meas_range_current` (auto/limited/fixed)

> 원본 `measure_iv`/`measure_sampling` 은 그대로 남아있어 기존 셀들도 문제없이 동작한다.

In [ ]:
def _sweep_mode(direction, spacing):
    """UI Direction(single/double) + Linear/Log -> pymeasure SweepMode 이름."""
    table = {
        ("single", "linear"): "LINEAR_SINGLE",
        ("double", "linear"): "LINEAR_DOUBLE",
        ("single", "log"):    "LOG_SINGLE",
        ("double", "log"):    "LOG_DOUBLE",
    }
    key = (direction.lower(), spacing.lower())
    if key not in table:
        raise ValueError(f"direction/spacing 조합 오류: {direction}/{spacing}")
    return table[key]


def _range_name(mode, rng):
    """UI Ranging Mode(auto/limited/fixed)+레인지값 -> meas_range_current 이름."""
    m = mode.lower()
    if m == "auto":
        return "Auto Ranging"
    if m == "limited":
        return f"{rng} limited auto ranging"
    if m == "fixed":
        return f"{rng} range fixed"
    raise ValueError(f"ranging mode 오류: {mode!r} (auto|limited|fixed 중 하나)")


def _var_points(cfg):
    """points 우선. step 이 지정되면 No of Step = round(|stop-start|/step)+1 로 환산."""
    if cfg.get("step"):
        return int(round(abs(cfg["stop"] - cfg["start"]) / cfg["step"])) + 1
    return int(cfg["points"])


def _var_values(cfg):
    """VAR2 스텝 전압 리스트 (선형). start > stop 이면 내림차순."""
    n = _var_points(cfg)
    if n <= 1:
        return [cfg["start"]]
    return [cfg["start"] + (cfg["stop"] - cfg["start"]) * k / (n - 1) for k in range(n)]


def _comp(value):
    """compliance 값 -> ramp_source 인자. None 이면 '' (장비 이전 설정 유지)."""
    return "" if value in (None, "") else value


def measure_iv_full(b1500, config=None, var1=None, var2="__default__",
                    timing=None, adc=None, ranges=None):
    """EasyEXPERT 'Measurement Setup'(VAR1/VAR2/Timing/Constants/Ranging)을 코드로 재현한 I-V 측정.

    인자는 None(또는 var2 는 "__default__") 이면 위 CONFIG 전역값을 사용. 개별 오버라이드 가능.
    - config : SMU_CONFIG (drive=sweep/const/off, const_v, compliance) — 역할 + Constants 정의
    - var1   : VAR1_CONFIG — 주 sweep 상세
    - var2   : VAR2_CONFIG or None — 부 sweep. None 이면 VAR2 미사용(=VAR1 1회)
    - timing/adc/ranges : TIMING/ADC/RANGE_CONFIG
    반환: DataFrame. VAR2 사용 시 맨 앞에 VAR2 값 컬럼이 붙고 스텝별 결과가 세로로 이어붙음.
    """
    config = SMU_CONFIG   if config is None else config
    var1   = VAR1_CONFIG  if var1   is None else var1
    var2   = VAR2_CONFIG  if var2 == "__default__" else var2
    timing = TIMING_CONFIG if timing is None else timing
    adc    = ADC_CONFIG   if adc    is None else adc
    ranges = RANGE_CONFIG if ranges is None else ranges

    sweep_ch = var1["unit"]
    if config.get(sweep_ch, {}).get("drive") != "sweep":
        raise ValueError(f"VAR1 unit(SMU{sweep_ch}) 은 SMU_CONFIG 에서 drive='sweep' 이어야 함.")

    const_chs = [ch for ch, c in config.items() if c["drive"] == "const"]
    active    = [sweep_ch] + const_chs
    smus      = {ch: getattr(b1500, f"smu{ch}") for ch in active}

    mode = _sweep_mode(var1["direction"], var1["spacing"])
    nop  = _var_points(var1)
    npts = nop if "SINGLE" in mode else 2 * nop      # 왕복(DOUBLE)이면 포인트 2배

    # VAR2 준비 (없으면 [None] 로 1회만)
    if var2:
        var2_ch = var2["unit"]
        if var2_ch not in const_chs:
            raise ValueError(f"VAR2 unit(SMU{var2_ch}) 은 SMU_CONFIG 에서 drive='const' 여야 함.")
        v2_vals = _var_values(var2)
    else:
        var2_ch, v2_vals = None, [None]

    frames = []
    for v2 in v2_vals:
        # --- 측정 모드/채널 (MM) ---
        b1500.meas_mode("STAIRCASE_SWEEP", *[smus[ch] for ch in active])
        for ch in active:
            s = smus[ch]
            s.enable()
            s.adc_type     = adc["adc_type"]
            s.meas_op_mode = "COMPLIANCE_SIDE"

        # --- Ranging Mode (RI) : RANGE_CONFIG 있는 채널만 ---
        for ch in active:
            rc = (ranges or {}).get(ch)
            if rc:
                smus[ch].meas_range_current = _range_name(rc["mode"], rc.get("range"))

        # --- ADC / Timing ---
        b1500.adc_setup(adc["adc_type"], adc["mode"], adc["N"])
        b1500.adc_auto_zero = adc.get("auto_zero", True)      # AZ (Auto-zero on/off)
        b1500.sweep_timing(timing["hold"], timing["delay"], step_delay=timing["step_delay"])
        b1500.sweep_auto_abort(timing["auto_abort"], post=timing["post"])

        # --- VAR1 sweep 소스 (WV) ---
        pcomp = "" if var1.get("power_comp") in (None, "") else var1["power_comp"]
        smus[sweep_ch].staircase_sweep_source(
            "VOLTAGE", mode, "Auto Ranging",
            var1["start"], var1["stop"], nop, var1["compliance"], pcomp,
        )
        # --- Constants (DV) : VAR2 채널이면 이번 스텝 값으로 덮어씀 ---
        #     compliance 는 VAR2 채널이면 VAR2_CONFIG, 아니면 SMU_CONFIG 값을 쓴다.
        for ch in const_chs:
            if ch == var2_ch:
                val  = v2
                comp = var2.get("compliance", config[ch].get("compliance"))
            else:
                val  = config[ch]["const_v"]
                comp = config[ch].get("compliance")
            smus[ch].ramp_source("VOLTAGE", "Auto Ranging", val, _comp(comp),
                                 stepsize=0.1, pause=20e-3)

        # --- 측정 실행 ---
        b1500.check_errors()
        b1500.clear_buffer()
        b1500.clear_timer()
        b1500.send_trigger()
        wait_measurement(b1500)
        data = b1500.read_data(npts)

        if var2_ch is not None:
            data.insert(0, f"VAR2 SMU{var2_ch} {var2.get('name','V')} (V)", v2)
        frames.append(data)

    # 모든 const SMU 0V 복귀
    for ch in const_chs:
        smus[ch].ramp_source("VOLTAGE", "Auto Ranging", 0, _comp(config[ch].get("compliance")),
                             stepsize=0.1, pause=20e-3)

    return pd.concat(frames, ignore_index=True) if len(frames) > 1 else frames[0]

## 2.5 (선택) Sampling 측정 — 시간에 따른 전류 (transient / 안정성)

`measure_iv` 는 전압을 **쓸면서**(sweep) I-V 곡선을 얻고, `measure_sampling` 은 전압을 **고정**한 채 일정 시간 간격마다 전류를 찍어 **시간축(I-t) 데이터**를 얻습니다. (B1500 매뉴얼의 `SAMPLING` 모드)

- 같은 `SMU_CONFIG` 를 그대로 재사용 — `sweep` 역할 SMU(GATE)는 `V_STOP`(또는 `SAMP_SWEEP_BIAS`)로 고정, `const` 역할은 `const_v` 로 고정, `off` 는 미사용.
- 반환 `DataFrame` 에는 **time stamp 열 + 채널별 전류 열**이 들어 있어 `전류 vs 시간` 으로 바로 그릴 수 있음.
- 용도: bias-stress 안정성, 접촉 안정화 확인, drift 모니터링 등. **필요 없으면 호출 안 하면 됨** (I-V 측정엔 영향 없음).

> ⚠️ `measure_iv`(STAIRCASE_SWEEP) 와 `measure_sampling`(SAMPLING) 은 측정 모드를 서로 바꾸므로, 한 좌표에서 **둘 다** 쓸 경우 순서대로(예: I-V 먼저 → sampling 나중) 호출하면 됩니다. 각자 자기 모드를 다시 설정하므로 충돌 없음.

In [ ]:
# --- Sampling(시간) 측정 파라미터 (필요할 때만) -------------------------------
# I-V(sweep)와 달리 전압을 '고정'해두고 일정 시간 간격마다 전류를 찍어
# 시간에 따른 변화(transient / bias-stress / 안정성)를 본다.
SAMP_INTERVAL  = 0.01    # 샘플 간격 [s]  (>=0.002 권장, 그 미만은 고속/제약 있음)
SAMP_NUMBER    = 200     # 샘플 개수  (총 측정시간 ≈ INTERVAL * NUMBER)
SAMP_HOLD_BIAS = 0       # bias 인가 후 첫 측정까지 대기시간 [s]
SAMP_BASE_V    = 0.0     # base(측정 전/후) 전압 [V]

# sweep 역할 SMU(예: GATE)를 sampling 중엔 어떤 DC 전압으로 고정할지.
# None 이면 V_STOP('on' 전압)을 사용. const 역할은 SMU_CONFIG 의 const_v 그대로.
SAMP_SWEEP_BIAS = None   # 예: 1.0 으로 두면 게이트를 1V 고정한 채 시간 측정


def measure_sampling(b1500, config, interval=SAMP_INTERVAL, number=SAMP_NUMBER,
                     hold_bias=SAMP_HOLD_BIAS, base=SAMP_BASE_V,
                     sweep_bias=SAMP_SWEEP_BIAS, compliance=I_COMPLIANCE):
    """SAMPLING 모드: 각 SMU(=팁)에 DC bias 를 걸고 '시간에 따른 전류'를 측정.
    - sweep 역할 SMU(GATE 등) → sweep_bias(None 이면 V_STOP)로 고정
    - const 역할 SMU         → SMU_CONFIG 의 const_v 로 고정
    - off                    → 사용 안 함
    반환: DataFrame (number 포인트, time stamp + 채널별 전류).
    measure_iv 와 같은 SMU_CONFIG 를 그대로 재사용한다."""
    sweep_chs = [ch for ch, c in config.items() if c["drive"] == "sweep"]
    const_chs = [ch for ch, c in config.items() if c["drive"] == "const"]
    active    = sweep_chs + const_chs
    if not active:
        raise ValueError("sampling 할 SMU 가 없습니다. SMU_CONFIG 확인.")

    smus = {ch: getattr(b1500, f"smu{ch}") for ch in active}

    # 측정 모드 = SAMPLING (측정 순서 = active 순서)
    b1500.meas_mode("SAMPLING", *[smus[ch] for ch in active])
    for ch in active:
        s = smus[ch]
        s.enable()
        s.adc_type = "HSADC"            # sampling 은 고속 ADC 사용
        s.meas_range_current = "1 uA"
        s.meas_op_mode = "COMPLIANCE_SIDE"

    b1500.sampling_mode = "LINEAR"
    b1500.adc_setup("HSADC", "AUTO", 1)
    b1500.sampling_timing(hold_bias, interval, number)   # MT: hold, interval, #points
    b1500.sampling_auto_abort(False, post="Bias")        # 중도중단 off
    b1500.time_stamp = True                              # 시간축 기록 ON

    # 각 SMU 에 DC bias 인가 (base → bias)
    for ch in active:
        bias = (sweep_bias if sweep_bias is not None else V_STOP) if ch in sweep_chs \
            else config[ch]["const_v"]
        smus[ch].sampling_source("VOLTAGE", "Auto Ranging", base, bias, compliance)

    # 측정 시작
    b1500.check_errors()
    b1500.clear_buffer()
    b1500.clear_timer()
    b1500.send_trigger()

    # 끝날 때까지 대기 후 한 번에 읽기 (time stamp + current 포함)
    wait_measurement(b1500)
    data = b1500.read_data(number)

    # 모든 SMU 0V 로 복귀
    for ch in active:
        smus[ch].ramp_source("VOLTAGE", "Auto Ranging", 0, stepsize=0.1, pause=20e-3)
    return data

## 3. 장비 연결

오류가 자주 나는데, vscode를 껐다 키고, B1500을 껐다 키고, 잭 연결이 잘 되어있는지 확인해보기

In [ ]:
s300 = CascadeS300(S300_GPIB)
print("S300:", s300.ask("*IDN?"))
print("S300 setup:", s300.setup())   # $:set:mode summit + $:set:resp on + REMOTE + metric (순서 중요)

b1500 = setup_b1500()
print("B1500:", b1500.ask("*IDN?"))

## 3.5 통신 확인 (S300 ↔ B1500)
좌표 이동(S300)·측정(B1500) 두 장비가 모두 응답하는지, S300 이 REMOTE/정렬 완료 상태인지 먼저 확인합니다.
**[OK] 가 떠야 다음으로 진행하세요.** (매뉴얼 p.9 `Verifying GPIB Communication` 절에 해당)

In [ ]:
def check_comm(s300, b1500):
    """S300(좌표이동) + B1500(측정) 통신/준비상태 확인."""
    print("=== 통신 확인 ===")
    ok = True

    # --- S300 ---
    idn = s300.ask("*IDN?")
    print("S300 *IDN?        :", idn)
    ok &= "Cascade" in idn

    mode = s300.ask("$:set:mode?")          # SUMMIT/EG = Nucleus 인터프리터 동작중
    print("S300 interpreter  :", mode)
    ok &= mode in ("SUMMIT", "EG")

    tst = s300.ask("*tst?")                 # 0 = self-test 정상
    print("S300 self-test    :", tst)
    ok &= tst.strip() == "0"

    s300.set_remote()
    oper = s300.ask(":SYST:OPER:MODE?")     # REMOTE 여야 원격제어 가능
    print("S300 oper mode    :", oper)
    ok &= oper == "REMOTE"

    align = s300.ask(":align:wafer:busy?")  # SUCCESS = 수동 alignment 끝난 상태인지 확인용
    print("S300 align status :", align)     # (정렬 안 됐으면 좌표 이동이 어긋남)

    # --- B1500 ---
    bidn = b1500.ask("*IDN?")
    print("B1500 *IDN?       :", bidn)
    ok &= "B1500" in bidn

    opc = b1500.ask("*OPC?")                # 1 = idle/완료
    print("B1500 *OPC?       :", opc)
    ok &= opc.strip() == "1"

    print("B1500 UNT?        :", b1500.ask("UNT?"))   # 장착 모듈 목록

    print("\n결과:", "[OK] 통신 정상" if ok else "[FAIL] 확인 필요 - 위 항목 점검")
    return ok


check_comm(s300, b1500)

## 4. 좌표 파일 읽기 (CSV / 엑셀 자동 판별)
`COORD_CSV` 의 확장자가 `.csv` 면 `read_csv`, `.xlsx/.xls` 면 `read_excel` 로 읽습니다. 둘 다 `Subsite Name, X Position, Y Position` 컬럼 구조는 동일.

In [ ]:
# 확장자 보고 CSV / 엑셀 자동 판별 (.xlsx 는 openpyxl 필요: pip install openpyxl)
ext = os.path.splitext(COORD_CSV)[1].lower()
if ext in (".xlsx", ".xls"):
    coords = pd.read_excel(COORD_CSV)
else:
    coords = pd.read_csv(COORD_CSV)

print(f"좌표 {len(coords)} 개 로드 완료: {COORD_CSV}")
coords.head()

## 5. 기준점 등록 — ⚠️ 사람이 첫 소자에 팁 contact 시킨 상태에서 실행
위 '준비' 단계대로 **사람이 첫 소자에 팁을 직접 contact** 시킨 상태에서 아래 셀을 실행하세요.
현재 위치가 모든 좌표의 원점(0,0)·contact 높이로 등록됩니다 (실제 contact 위치 기준이라 이후 소자를 찍지 않음).

In [ ]:
s300.set_reference()

## 6. 메인 루프 — 좌표대로 쭉 이동 + 측정 (① + ② 합치는 곳)

**좌표 convention**: 파일의 X/Y 는 **원점(사람이 contact시킨 소자=0,0) 기준 상대좌표**이고 음수도 가능 (예: 좌상단 `-100, 100`). 보통은 처음 찍는 소자를 `0,0` 으로 둔다.

- **좌표 (0,0)** 인 줄 = 원점 = 사람이 이미 contact → 이동 없이 바로 측정
- **그 외** = `separate()`(분리) → `move_xy()`(XY 이동, Z 자동 분리) → `contact()`(접촉) → 측정

> 💡 처음엔 측정 없이 **이동만** 확인하려면 아래 `measure_iv(...)` 와 저장 줄을 주석 처리하세요.

In [ ]:
os.makedirs(OUT_DIR, exist_ok=True)

# --- 팁 보호: Z 축만 느리게 (XY 속도는 건드리지 않음 = 장비 기본값 유지) ---
s300.cmd(":set:cont:spee 100")              # 접촉(올림) 100 µm/s (기본 500 → 느리게)
s300.cmd(f":set:vel {CHUCK_ID} z slow")     # z(내림/복귀)만 느리게. xy 는 손대지 않음
print("팁보호 속도: 접촉", s300.ask(":set:cont:spee?"), "µm/s, z=slow (xy 기본값 유지)")

# 측정 모드 선택: "iv"=I-V만 | "sampling"=시간측정만 | "both"=둘 다
MEASURE_MODE = "iv"

assert MEASURE_MODE in ("iv", "sampling", "both"), 'MEASURE_MODE 는 "iv"/"sampling"/"both" 중 하나'
do_iv   = MEASURE_MODE in ("iv", "both")
do_samp = MEASURE_MODE in ("sampling", "both")
print(f"측정 모드: {MEASURE_MODE}  (I-V={do_iv}, sampling={do_samp})")

# ⚠️ 이동거리 배율 (테스트용). test_coordinates.csv 는 100µm라 눈에 안 보여서 확대해 확인.
#    실제 좌표 파일(grid_coordinates.csv)로 측정할 땐 반드시 1 로 되돌릴 것!
COORD_SCALE = 1     # 실제 측정: CSV 좌표 그대로 사용 (테스트 땐 100 이었음)
print(f"이동거리 배율: ×{COORD_SCALE}")
LIFT_Z = 2000   # 지점 간 이동 시 척을 "아래로" 내릴 분리량 [µm] (2mm)


for _, row in coords.iterrows():
    sub_id = row["Subsite Name"]
    dx = int(row["X Position"]) * COORD_SCALE   # 원점 기준 상대좌표 × 배율 (음수 가능)
    dy = int(row["Y Position"]) * COORD_SCALE

    if dx == 0 and dy == 0:
        # 좌표 (0,0) = 원점 = 사람이 이미 contact 시킨 소자 → 이동 없이 바로 측정
        print(f"[Subsite {sub_id}] 원점 (0, 0) — 이동 없음")
    else:
        print(f"[Subsite {sub_id}] -> 원점+({dx}, {dy}) 이동")
        # 확정 방향: z↑=척 위(팁 쪽) / z↓=척 아래(팁에서 멀어짐)
        # 순서: ① 분리(척 아래로) → ② XY 이동(분리된 채) → ③ 접촉(척 위로)
        xc, yc, zc = s300.read_position()
        s300.cmd(f":mov:abs {CHUCK_ID} {int(xc)} {int(yc)} {int(zc) - LIFT_Z}")  # ① 분리: 아래로(z↓)
        print("    ① 분리(척 아래로):", s300.read_position())
        s300.move_xy(dx, dy)                                                     # ② XY 이동 (분리된 채)
        print("    ② 이동:", s300.read_position())
        s300.contact()                                                          # ③ 접촉 (척 위로 올려 팁 닿음)
        print("    ③ 접촉(척 위로):", s300.read_position())
        time.sleep(0.2)



    # ① I-V 측정 (STAIRCASE_SWEEP) — 팁 역할은 SMU_CONFIG 따름
    if do_iv:
        data = measure_iv(b1500, V_START, V_STOP, V_POINTS, I_COMPLIANCE, SMU_CONFIG)
        out_path = os.path.join(OUT_DIR, f"subsite_{sub_id}.csv")
        data.to_csv(out_path)
        print(f"  저장(I-V): {out_path}")

    # ② sampling 측정 (SAMPLING, 시간축) — 같은 SMU_CONFIG 재사용, 별도 파일
    if do_samp:
        samp = measure_sampling(b1500, SMU_CONFIG)
        samp_path = os.path.join(OUT_DIR, f"subsite_{sub_id}_sampling.csv")
        samp.to_csv(samp_path)
        print(f"  저장(sampling): {samp_path}")

## 6-B. (대안) 메인 루프 — `measure_iv_full()` 사용  ← **전달특성은 이걸 쓴다**

위 `## 6` 원본 루프는 `measure_iv`(하드코딩)를 쓴다. 아래 루프는 **똑같은 좌표이동**에
측정만 `measure_iv_full`(설정판 반영)로 바꾼 것. 둘 중 하나만 실행하면 된다.

- `VAR1_CONFIG`(Vg sweep) 로 sweep 조건 결정. `VAR2_CONFIG`(Vd step) 지정 시 좌표당 2차원 sweep.
- 결과는 `results_idvg/` 에 저장 (Id-Vd 결과 `results_idvd/` 와 분리).


In [ ]:
OUT_DIR_CFG = "results_idvg"   # Id-Vg(전달특성) 결과 폴더 (Id-Vd 결과와 분리)
os.makedirs(OUT_DIR_CFG, exist_ok=True)

# --- 팁 보호: 접촉(올라오는) 동작을 소프트웨어로 잘라서 천천히 -----------------
# 실측 결과 :set:cont:spee 는 :mov:cont 전 구간 속도를 지배하지 않는다.
#   (25 µm/s 로 설정해도 1000 µm 를 0.28 s = 약 3500 µm/s 로 이동)
# 그래서 분리높이 -> 접촉높이 구간을 :mov:abs 로 Z_STEP 씩 나눠 올린다.
SLOW_CONTACT = True    # False 면 그냥 s300.contact() (빠름)
Z_STEP  = 25           # 한 스텝에 올릴 높이 [µm]  (작을수록 느림)
Z_PAUSE = 0.1          # 스텝 사이 대기 [s]        (클수록 느림)
Z_FINAL = 50           # 마지막 이 구간은 contact() 에 맡김 [µm]
                       #   → 등록된 접촉 높이에서 정확히 멈추는 것을 보장

s300.cmd(f":set:vel {CHUCK_ID} z slow")     # z 만 느리게. xy 는 손대지 않음


def contact_slow(z_contact):
    """분리 상태에서 z_contact 까지 Z_STEP 씩 천천히 올린 뒤 마지막만 contact() 로 마무리.

    z_contact : 등록된 접촉 높이 [µm] (분리 직전에 읽어둔 z)
    마지막 Z_FINAL 구간을 contact() 에 맡기므로 등록 높이 위로는 절대 올라가지 않는다.
    """
    x, y, z = (int(v) for v in s300.read_position())
    z_stop = int(z_contact) - Z_FINAL
    while z < z_stop:
        z = min(z + Z_STEP, z_stop)
        s300.cmd(f":mov:abs {CHUCK_ID} {x} {y} {z}")
        time.sleep(Z_PAUSE)
    return s300.contact()


def do_contact(z_contact):
    """SLOW_CONTACT 설정에 따라 천천히/기본 접촉."""
    return contact_slow(z_contact) if SLOW_CONTACT else s300.contact()


_n1 = _var_points(VAR1_CONFIG)
_n2 = _var_points(VAR2_CONFIG) if VAR2_CONFIG else 1
_pts = (_n1 if VAR1_CONFIG["direction"] == "single" else 2 * _n1) * _n2
print("VAR1:", VAR1_CONFIG["name"],
      f'{VAR1_CONFIG["start"]}~{VAR1_CONFIG["stop"]}V',
      f'{_n1}pts', VAR1_CONFIG["direction"], VAR1_CONFIG["spacing"],
      f'comp={VAR1_CONFIG["compliance"]}A')
print("VAR2:", "미사용" if not VAR2_CONFIG else
      f'{VAR2_CONFIG["name"]} {VAR2_CONFIG["start"]}~{VAR2_CONFIG["stop"]}V '
      f'{_n2}스텝 comp={VAR2_CONFIG["compliance"]}A')
print(f"소자당 {_pts} 점 x {len(coords)} 소자 → 저장 폴더: {OUT_DIR_CFG}/")
print(f"접촉 방식: {'천천히 (%d µm/스텝, %.2fs 대기)' % (Z_STEP, Z_PAUSE) if SLOW_CONTACT else '기본 contact()'}")

# 시작 시점 = 원점 소자에 컨택된 상태. 이 높이를 마지막 원점 복귀에 재사용한다.
Z_HOME = int(s300.read_position()[2])
print("접촉 높이(z):", Z_HOME)

for _, row in coords.iterrows():
    sub_id = row["Subsite Name"]
    dx = int(row["X Position"])
    dy = int(row["Y Position"])

    if dx == 0 and dy == 0:
        # 좌표 (0,0) = 원점 = 사람이 이미 contact 시킨 소자 → 이동 없이 바로 측정
        print(f"[Subsite {sub_id}] 원점 (0, 0) — 이동 없음")
    else:
        print(f"[Subsite {sub_id}] -> 원점+({dx}, {dy}) 이동")
        z_contact = int(s300.read_position()[2])   # 지금 = 등록된 접촉 높이
        s300.separate()                            # ① 팁 분리 (아래로, 팁에서 멀어짐)
        s300.move_xy(dx, dy)                       # ② XY 이동 (분리된 채)
        t0 = time.time()
        do_contact(z_contact)                      # ③ 접촉
        print(f"    위치: {s300.read_position()}  (접촉 {time.time() - t0:.1f} s)")
        time.sleep(0.2)

    # Id-Vg 측정 (VAR1=Vg sweep, VAR2=Vd step) — ## 0-B 설정 그대로
    t0 = time.time()
    data = measure_iv_full(b1500)
    out_path = os.path.join(OUT_DIR_CFG, f"subsite_{sub_id}.csv")
    data.to_csv(out_path, index=False)
    print(f"  저장: {out_path}  ({len(data)} 행, {time.time() - t0:.1f} s)")

# --- 마무리: 분리 -> 원점(0,0) 복귀 -> 다시 접촉 -----------------------------
# 마지막에 원점에 컨택된 상태로 끝내야, 이 셀을 다시 돌릴 때 첫 소자(0,0)를
# 곧바로 측정하는 전제가 그대로 성립한다.
print("원점 복귀 중...")
s300.separate()
s300.move_xy(0, 0)
do_contact(Z_HOME)
print("원점 복귀 완료:", s300.read_position())
print("전체 측정 완료.")

## 7. 마무리 — 팁 분리 후 원점 복귀

In [ ]:
s300.separate()      # 팁 분리
s300.move_xy(0, 0)   # 원점으로 복귀 (Z 자동 분리 상태로 이동)
print("측정 완료.")

In [ ]:
# --- 종료: S300 LOCAL 복귀 + 세션 정리 -------------------------------------
# 케이블 뽑기/세션 끝내기 전에 실행. (척을 움직이지 않는 안전한 명령)
print("S300 -> LOCAL:", s300.ask(":SYST:OPER:MODE LOCAL"))   # 원격모드 해제 (UI 수동조작 복귀)

# VISA 세션 닫기 (생략해도 커널 Restart 시 자동 해제됨)
try:
    s300.instrument.close()
    b1500.adapter.close()
    print("VISA 세션 닫음.")
except Exception as e:
    print("close 중 예외(무시 가능):", e)


In [ ]:
import time
D = 10000   # 이동 거리 [µm] = 1cm  (더 키우려면 이 숫자만 바꾸기)
print("시작:", s300.read_position())
for target in [(D, 0), (D, D), (0, D), (0, 0)]:
    print(f"→ 이동 {target}:", s300.move_xy(*target))
    time.sleep(2)   # 각 모서리 2초 멈춤 — 눈으로 확인


In [ ]:
import time, os
os.makedirs(OUT_DIR, exist_ok=True)

D = 10000   # 이동 거리 [µm] = 1cm  (숫자만 바꾸면 조절)
targets = [(0, 0), (D, 0), (D, D), (0, D), (0, 0)]   # 원점→오른쪽→위→왼쪽→원점

print("시작 위치:", s300.read_position())
for i, (dx, dy) in enumerate(targets):
    pos = s300.move_xy(dx, dy)                 # ① XY 이동 (팁 비접촉, Z는 안전높이)
    print(f"[{i}] 이동 {(dx, dy)} → 실제위치 {pos}")

    data = measure_iv(b1500, V_START, V_STOP, V_POINTS, I_COMPLIANCE, SMU_CONFIG)  # ② 측정 (개방→≈0)
    path = os.path.join(OUT_DIR, f"movemeas_{i}_{dx}_{dy}.csv")
    data.to_csv(path)
    print(f"     측정 저장: {path}  (전류≈0 예상, {len(data)}포인트)")

    time.sleep(1)   # 눈으로 확인용 잠깐 멈춤
print("완료")


In [ ]:
# ── Z축 이동 테스트 : 팁에서 300µm 멀어졌다 복귀 ──────────────────────
#  이 프로버는 z 가 거꾸로: z 증가 = 척 아래로(팁에서 멀어짐) = ✅안전
#                          z 감소 = 팁 쪽 = ⚠️위험 (쓰지 말 것)

def move_z_abs(z):
    """현재 X,Y 유지한 채 Z만 절대이동 (:mov:abs 2 x y z). 완료까지 대기."""
    x, y, _ = s300.read_position()
    s300.cmd(f":mov:abs {CHUCK_ID} {int(x)} {int(y)} {int(z)}")
    return s300.read_position()

DZ = 300   # 팁에서 멀어질 양 [µm] = 0.3mm (멀어지는 방향이라 커도 안전)

x0, y0, z0 = s300.read_position()
print("시작            :", (x0, y0, z0))
print(f"→ 팁에서 {DZ}um 멀어짐:", move_z_abs(z0 + DZ))   # +z = 멀어짐(안전)
time.sleep(1)                                            # 눈으로 확인
print("→ 원위치 복귀    :", move_z_abs(z0))               # 시작 높이로
print("Z 테스트 완료.")




In [ ]:
import time

# 확인됨(실측): z 증가 = 척 위로 / z 감소 = 척 아래로
DOWN = -1   # 아래로 = z 감소

x, y, z = (int(v) for v in s300.read_position())
print(f"시작 z = {z} — 척을 아래로 최대한 내립니다")

for step in (2000, 500, 100):          # 큰 폭 → 점점 작게 (리밋에 최대한 근접)
    while True:
        try:
            s300.cmd(f":mov:abs {CHUCK_ID} {x} {y} {z + DOWN*step}")
        except RuntimeError:
            break                       # 이 폭으론 더 못 감 → 다음(더 작은) 폭으로
        z = int(s300.read_position()[2])
        print(f"   (step {step}) z = {z}")
        time.sleep(0.2)

print("최하단 도달. z =", s300.read_position()[2])


In [ ]:
x, y, z = s300.read_position()
print(f"현재 위치: X={x}, Y={y}, Z={z}")
print(f"현재 Z = {z} µm")


In [ ]:
x, y, z = (int(v) for v in s300.read_position())
print("시작 z =", z)

s300.cmd(f":mov:abs {CHUCK_ID} {x} {y} 5000")   # Z를 5000 으로 (XY 그대로)

print("이동후 z =", s300.read_position()[2])


In [ ]:
# ── 이동 없이 "지금 접촉된 자리"만 측정 + 저장 ─────────────────────────
#  사람이 팁을 접촉시켜 둔 상태 그대로. XY/Z 이동 명령 전혀 없음.

import os
from datetime import datetime

SAMPLE_NAME = "sample1"           # 파일명에 들어갈 이름 (원하는 대로 변경)
OUT_DIR_ONE = "results_single_idvg"   # 저장 폴더 (Id-Vd 파일과 섞이지 않게 분리)
os.makedirs(OUT_DIR_ONE, exist_ok=True)

print("현재 위치(이동 안 함):", s300.read_position())   # 확인용 조회만 (움직이지 않음)

# 측정 — 설정 셀의 V_START/V_STOP/V_POINTS/I_COMPLIANCE/SMU_CONFIG 그대로 사용
data = measure_iv(b1500, V_START, V_STOP, V_POINTS, I_COMPLIANCE, SMU_CONFIG)

# 저장 (실행할 때마다 시각이 붙어 덮어쓰지 않음)
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
path = os.path.join(OUT_DIR_ONE, f"{SAMPLE_NAME}_{stamp}.csv")
data.to_csv(path)

print(f"저장 완료: {path}   ({len(data)} 행)")
print("컬럼:", list(data.columns))
data.head()


In [ ]:
print(s300.ask(":SYST:OPER:MODE LOCAL"))

In [ ]:
import time

Z_STEP, Z_PAUSE, Z_FINAL = 25, 0.1, 50

z_contact = int(s300.read_position()[2])
print("현재:", s300.read_position(), " 접촉 높이 z =", z_contact)

s300.separate()
print("① 분리:", s300.read_position())

s300.move_xy(0, 0)
print("② 원점 이동:", s300.read_position())

x, y, z = (int(v) for v in s300.read_position())
z_stop = z_contact - Z_FINAL
t0 = time.time()
while z < z_stop:
    z = min(z + Z_STEP, z_stop)
    s300.cmd(f":mov:abs {CHUCK_ID} {x} {y} {z}")
    time.sleep(Z_PAUSE)
s300.contact()
print(f"③ 접촉: {s300.read_position()}  ({time.time() - t0:.1f} s)")


In [ ]:
import pyvisa, time
rm = pyvisa.ResourceManager(VISA_LIB)
print("1차:", rm.list_resources())

intf = rm.open_resource("GPIB0::INTFC")
intf.send_ifc()          # Interface Clear
intf.close()
time.sleep(0.5)
print("IFC 후:", rm.list_resources())


In [ ]:
import time

def measure_iv_fast(b1500, config=None, var1=None, var2="__default__",
                    timing=None, adc=None, ranges=None, verbose=True):
    """measure_iv_full 과 측정 조건은 동일. 설정 전송을 VAR2 루프 밖으로 빼고
    스텝마다는 게이트 전압(DV)만 바꾼다. 구간별 소요시간을 같이 출력."""
    config = SMU_CONFIG   if config is None else config
    var1   = VAR1_CONFIG  if var1   is None else var1
    var2   = VAR2_CONFIG  if var2 == "__default__" else var2
    timing = TIMING_CONFIG if timing is None else timing
    adc    = ADC_CONFIG   if adc    is None else adc
    ranges = RANGE_CONFIG if ranges is None else ranges

    sweep_ch  = var1["unit"]
    const_chs = [ch for ch, c in config.items() if c["drive"] == "const"]
    active    = [sweep_ch] + const_chs
    smus      = {ch: getattr(b1500, f"smu{ch}") for ch in active}

    mode = _sweep_mode(var1["direction"], var1["spacing"])
    nop  = _var_points(var1)
    npts = nop if "SINGLE" in mode else 2 * nop

    if var2:
        var2_ch = var2["unit"]
        v2_vals = _var_values(var2)
        v2_comp = var2.get("compliance", config[var2_ch].get("compliance"))
    else:
        var2_ch, v2_vals, v2_comp = None, [None], None

    # ---------- 설정: 여기서 딱 한 번만 ----------
    t = time.time()
    b1500.meas_mode("STAIRCASE_SWEEP", *[smus[ch] for ch in active])
    for ch in active:
        s = smus[ch]
        s.enable()
        s.adc_type     = adc["adc_type"]
        s.meas_op_mode = "COMPLIANCE_SIDE"
        rc = (ranges or {}).get(ch)
        if rc:
            s.meas_range_current = _range_name(rc["mode"], rc.get("range"))
    b1500.adc_setup(adc["adc_type"], adc["mode"], adc["N"])
    b1500.adc_auto_zero = adc.get("auto_zero", True)
    b1500.sweep_timing(timing["hold"], timing["delay"], step_delay=timing["step_delay"])
    b1500.sweep_auto_abort(timing["auto_abort"], post=timing["post"])
    pcomp = "" if var1.get("power_comp") in (None, "") else var1["power_comp"]
    smus[sweep_ch].staircase_sweep_source(
        "VOLTAGE", mode, "Auto Ranging",
        var1["start"], var1["stop"], nop, var1["compliance"], pcomp,
    )
    for ch in const_chs:                      # SOURCE 등 고정 채널
        if ch == var2_ch:
            continue
        smus[ch].ramp_source("VOLTAGE", "Auto Ranging", config[ch]["const_v"],
                             _comp(config[ch].get("compliance")),
                             stepsize=0.1, pause=20e-3)
    if var2_ch is not None:                   # 게이트 첫 값까지는 램프로 안전하게
        smus[var2_ch].ramp_source("VOLTAGE", "Auto Ranging", v2_vals[0],
                                  _comp(v2_comp), stepsize=0.5, pause=20e-3)
    b1500.check_errors()
    t_setup = time.time() - t

    # ---------- VAR2 루프: 게이트 전압만 바꾸고 트리거 ----------
    frames = []
    tm = {"gate": 0.0, "trig": 0.0, "wait": 0.0, "read": 0.0}
    for i, v2 in enumerate(v2_vals):
        t = time.time()
        if var2_ch is not None and i > 0:
            smus[var2_ch].force("VOLTAGE", "Auto Ranging", v2, _comp(v2_comp))
        tm["gate"] += time.time() - t

        t = time.time()
        b1500.clear_buffer(); b1500.clear_timer(); b1500.send_trigger()
        tm["trig"] += time.time() - t

        t = time.time(); wait_measurement(b1500);      tm["wait"] += time.time() - t
        t = time.time(); data = b1500.read_data(npts); tm["read"] += time.time() - t

        if var2_ch is not None:
            data.insert(0, f"VAR2 SMU{var2_ch} {var2.get('name','V')} (V)", v2)
        frames.append(data)

    for ch in const_chs:                      # 0 V 복귀
        smus[ch].ramp_source("VOLTAGE", "Auto Ranging", 0,
                             _comp(config[ch].get("compliance")),
                             stepsize=0.5, pause=20e-3)

    if verbose:
        print(f"    [타이밍] 설정 {t_setup:.1f}s | 게이트 {tm['gate']:.1f}s | "
              f"트리거 {tm['trig']:.1f}s | 측정대기 {tm['wait']:.1f}s | 읽기 {tm['read']:.1f}s")
    return pd.concat(frames, ignore_index=True) if len(frames) > 1 else frames[0]


In [ ]:
t0 = time.time()
df = measure_iv_fast(b1500)
print(f"총 {time.time() - t0:.1f} s, {len(df)} 행")
df.head()


In [ ]:
out = os.path.join(OUT_DIR_CFG, "test_subsite1.csv")
os.makedirs(os.path.dirname(out), exist_ok=True)
df.to_csv(out, index=False)
print("저장:", out, len(df), "행")


In [ ]:
try:
    b1500.adapter.close()
except Exception as e:
    print(e)
try:
    s300.instrument.close()
except Exception as e:
    print(e)


## 8. (선택) 전달특성 곡선 확인 — log Id vs Vg

전달특성은 **로그 스케일**로 봐야 의미가 있다 (off 전류가 pA, on 전류가 mA 라 선형축에선 안 보임).
바로 위에서 측정한 `df` 를 그리거나, 저장된 CSV 를 읽어서 그린다.
측정 자체와는 무관한 확인용 셀이라 안 돌려도 된다.


In [ ]:
# --- 전달특성 곡선 그리기 (확인용) -----------------------------------------
import matplotlib.pyplot as plt

# 그릴 데이터: 위 셀에서 측정한 df, 또는 저장된 CSV
df_plot = df
# df_plot = pd.read_csv(os.path.join(OUT_DIR_CFG, "subsite_1.csv"))

print("컬럼:", list(df_plot.columns))


def _find_col(df, kind, ch):
    """pymeasure read_data 의 컬럼명은 버전에 따라 조금씩 달라서 부분일치로 찾는다.
    kind = 'Voltage' | 'Current', ch = SMU 채널 번호."""
    for c in df.columns:
        if kind.lower() in c.lower() and str(ch) in c:
            return c
    raise KeyError(f"SMU{ch} 의 {kind} 컬럼을 못 찾음. 실제 컬럼: {list(df.columns)}")


vg_col = _find_col(df_plot, "Voltage", VAR1_CONFIG["unit"])   # Gate 전압
id_col = _find_col(df_plot, "Current", VAR2_CONFIG["unit"] if VAR2_CONFIG else 2)  # Drain 전류
v2_col = next((c for c in df_plot.columns if c.startswith("VAR2")), None)

fig, ax = plt.subplots(figsize=(5, 4))
groups = df_plot.groupby(v2_col) if v2_col else [(None, df_plot)]
for v2, g in groups:
    label = f'{VAR2_CONFIG["name"]} = {v2} V' if v2 is not None else None
    ax.semilogy(g[vg_col], g[id_col].abs(), marker=".", ms=3, lw=1, label=label)

ax.set_xlabel(f'{VAR1_CONFIG["name"]} (V)')
ax.set_ylabel("|Id| (A)")
ax.set_title("Transfer curve (Id-Vg)")
ax.grid(True, which="both", alpha=0.3)
if v2_col:
    ax.legend()
plt.tight_layout()
plt.show()
